### Imports

In [1]:
import json
import os
import pandas as pd

from utils_MS import *

# %load_ext autotime

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

import plotly.graph_objects as go
import sys
import torch
import torch.nn.functional as F
import os.path as osp

%matplotlib inline
from IPython import display
from IPython.display import HTML
from matplotlib import animation

from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split

from torch_geometric.datasets import KarateClub, Planetoid
from torch_geometric.data import Data
from torch import nn, optim, Tensor
from torch_geometric.utils import structured_negative_sampling, to_networkx
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.nn import LGConv, Node2Vec, GCNConv, GATv2Conv, SAGEConv
from torch_geometric.nn import global_mean_pool, global_add_pool
from torch.nn import Linear, Sequential, BatchNorm1d, ReLU, Dropout
import torch_geometric.transforms as T

from tqdm import tqdm

/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.6.1.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, silhouette_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import kneighbors_graph

In [6]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE

In [7]:
# def run(params):

### Parameters

In [8]:
""" try:
    dir = os.path.dirname(os.path.abspath(__file__))
except:
    dir = os.getcwd()
print(dir) """

' try:\n    dir = os.path.dirname(os.path.abspath(__file__))\nexcept:\n    dir = os.getcwd()\nprint(dir) '

In [9]:
dict_dataset = {
    1: ["Mentos_2_process_NormalizationFiltered_format", ["Orange"]], # new mentos,  for Metabolomics
    2: ["deybis_filter_september_2br_3ar_format", ["SecoAmazonas"]],
    3: ["deybis_filter_december_2br_3ar_format", ["SecoAmazonas"]],
    4: ["deybis_filter_september_2br_10ar_format", ["SecoAmazonas"]],
    5: ["deybis_filter_december_2br_10ar_format", ["SecoAmazonas"]], # for Metabolomics
    6: ["deybis_filter_september_min_2br_3ar_format", ["SecoAmazonas"]],
    7: ["deybis_filter_december_min_2br_3ar_format", ["SecoAmazonas"]],
    8: ["vanessa_december_2br_3ar_format", ["SecoAmazonas"]],
    9: ["Pablo_2br_nar_format", ["AR"]],
    10: ["Pablo_2br_14ar_format", ["AR"]],
}
dataset = dict_dataset[9] # change
dataset

['Pablo_2br_nar_format', ['AR']]

In [10]:
params = {
    "exp": "exp0", # Change
    "methods": ["t-gae"], # ["vgae-base", "argva-base", "vgae-line", "dgi-tran", "t-gae"],
    "data_variations": ["none"],
    "has_transformation": True, # True or False
    "controls": dataset[1],
    "dimension": 64,
    "threshold_corr": 0.5,
    "threshold_log2": 0,
    "alpha": 0.05,
    "iterations": 1,
    "raw_data_file": dataset[0],
    "groups_id_no": ["Blank", "QC", "Std"],
    "sensitivity": False, # False: f1 (selectivity), True: f1 (selectivity), f2 (sensitivity)
    "obs": "",
    "seeds": [41, 42, 43, 44, 45, 46],
    
    "from": "python",
    "cuda": 0,
    "epochs": 100,
    "lr": 0.0001,
    "weight_decay": 1e-4,
    "patience": 10,
    "contamination": 0.1, # float in (0., 0.5)
    "n_jobs": 1, # -1 all
}

In [11]:
""" dir_path = "experiments/output"
res = sorted(os.listdir(dir_path))
n = len(res)
exp = "exp{}".format(n) """

exp = str(params["exp"])
exp

'exp0'

### Load dataset

In [12]:
exp = "exp9"
file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

exp = params["exp"]

print("Exp:\t\t", exp)

methods = params["methods"]
print("Methods:\t", methods)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

controls = params["controls"]
print("Control:\t", controls)

groups_id = params["groups_id"]
# groups_id = ['SecoAmazonas', 'SecoCusco', 'SecoSanMartin', 'FrescoAmazonas', 'FrescoCusco', 'FrescoSanMartin']
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups = params["groups"]
print("Groups:\t\t", groups)

# encoders = ["GIN", "GINE"]
encoders = ["GIN"]

Exp:		 exp9
Methods:	 ['t-gae']
Data variations: ['none']
Control:	 ['AR']
Groups id:	 ['AR', 'CRS', 'OSA', 'LPRD', 'SGB', 'LSNB', 'RCC', 'BC', 'BPH', 'PCa', 'PD']
Subgroups id:	 {'AR': ['1', '2'], 'CRS': ['1', '2'], 'OSA': ['1', '2'], 'LPRD': ['1', '2'], 'SGB': ['1', '2'], 'LSNB': ['1', '2'], 'RCC': ['1', '2'], 'BC': ['1', '2'], 'BPH': ['1', '2'], 'PCa': ['1', '2'], 'PD': ['1', '2']}
Groups:		 [['AR', 'CRS'], ['AR', 'OSA'], ['AR', 'LPRD'], ['AR', 'SGB'], ['AR', 'LSNB'], ['AR', 'RCC'], ['AR', 'BC'], ['AR', 'BPH'], ['AR', 'PCa'], ['AR', 'PD']]


In [13]:
params["raw_data_file"] = "Pablo_2br_nar_format" # Obs

In [14]:
# load dataset groups
if params["from"] == "python":
    df_raw = pd.read_csv("experiments/raw_data/{}.csv".format(params["raw_data_file"]), delimiter="|")
elif params["from"] == "drf":
    df_raw = pd.read_csv("{}".format(params["raw_data_file"]), delimiter="|") # from DRF
df_raw

,Alignment ID,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,...,PD_2.43,PD_2.44,PD_2.45,PD_2.46,PD_2.47,PD_2.48,PD_2.49,PD_2.50,PD_2.51,PD_2.52
0,0,1.0,69.99951,Unknown,0.575329,-2.215760,-1.877019,-1.518844,-1.398838,-1.298920,...,-1.096866,0.333022,0.273693,0.478082,-1.012425,-1.078037,0.566607,-1.074093,-1.041128,-1.076718
1,1,1.0,70.04025,Unknown,2.519698,0.524517,0.862641,1.220163,1.339951,1.439686,...,1.992662,2.054098,2.342706,2.060094,2.337721,2.048076,2.197014,2.063083,2.188817,2.051090
2,2,1.0,70.04151,Unknown,0.580982,-0.489549,-0.214833,0.075646,0.172970,0.254003,...,0.520551,0.533229,0.582630,0.534187,0.581785,0.532269,0.560055,0.535142,0.559160,1.820908
3,3,1.0,70.04908,Unknown,2.570505,-0.107955,0.215094,0.556676,0.671123,0.766412,...,1.365489,0.697392,2.546915,0.700091,2.183391,0.694677,2.494512,0.701435,0.730124,0.696036
4,4,1.0,70.06267,Unknown,1.157874,-0.151374,0.127539,0.297939,0.422454,0.521265,...,1.125370,1.149314,2.839522,2.768711,1.226690,1.147188,1.188020,2.990083,1.187039,1.148252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,5439,1.0,732.79951,Unknown,2.214770,-0.588492,-0.389296,-0.178671,-0.108101,4.236125,...,0.103094,0.175553,0.453542,0.181423,0.451090,0.169659,0.318208,0.184348,0.310187,0.172609
5440,5440,1.0,748.76437,Unknown,0.725796,-1.431324,-1.052855,-0.652672,-0.518591,-0.406954,...,0.197692,0.275529,0.580224,0.281838,0.574945,0.269194,0.431907,0.284983,0.423283,0.272365
5441,5441,1.0,794.79590,Unknown,1.651239,-1.204281,-0.800512,-0.373577,-0.230533,-0.111434,...,0.318278,0.382511,0.619537,0.388773,0.614171,0.376221,2.624637,0.391893,0.511456,0.379369
5442,5442,1.0,800.81295,Unknown,1.511482,-1.036480,-0.607684,-0.154288,-0.002378,0.124103,...,0.754284,0.812592,1.028620,1.608131,1.024227,0.807407,0.923866,0.817754,0.919121,0.810002


### Format dataset

In [15]:
# has transformation
""" columns_data = list(df_raw.columns)[4:]
if params["has_transformation"]:
    print("transformation")
    for column in columns_data:
        df_raw[column] = df_raw[column].apply(lambda x: 10**x)
df_raw """

' columns_data = list(df_raw.columns)[4:]\nif params["has_transformation"]:\n    print("transformation")\n    for column in columns_data:\n        df_raw[column] = df_raw[column].apply(lambda x: 10**x)\ndf_raw '

In [16]:
# concat
df_join_raw = pd.concat([
    df_raw.iloc[:, :]], axis=1)
df_join_raw.set_index("Alignment ID", inplace=True)
df_join_raw

,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,...,PD_2.43,PD_2.44,PD_2.45,PD_2.46,PD_2.47,PD_2.48,PD_2.49,PD_2.50,PD_2.51,PD_2.52
Alignment ID,,,,,,,,,,,,,,,,,,,,,
0,1.0,69.99951,Unknown,0.575329,-2.215760,-1.877019,-1.518844,-1.398838,-1.298920,-1.213064,...,-1.096866,0.333022,0.273693,0.478082,-1.012425,-1.078037,0.566607,-1.074093,-1.041128,-1.076718
1,1.0,70.04025,Unknown,2.519698,0.524517,0.862641,1.220163,1.339951,1.439686,1.525385,...,1.992662,2.054098,2.342706,2.060094,2.337721,2.048076,2.197014,2.063083,2.188817,2.051090
2,1.0,70.04151,Unknown,0.580982,-0.489549,-0.214833,0.075646,0.172970,0.254003,0.323632,...,0.520551,0.533229,0.582630,0.534187,0.581785,0.532269,0.560055,0.535142,0.559160,1.820908
3,1.0,70.04908,Unknown,2.570505,-0.107955,0.215094,0.556676,0.671123,0.766412,0.848291,...,1.365489,0.697392,2.546915,0.700091,2.183391,0.694677,2.494512,0.701435,0.730124,0.696036
4,1.0,70.06267,Unknown,1.157874,-0.151374,0.127539,0.297939,0.422454,0.521265,0.603536,...,1.125370,1.149314,2.839522,2.768711,1.226690,1.147188,1.188020,2.990083,1.187039,1.148252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,1.0,732.79951,Unknown,2.214770,-0.588492,-0.389296,-0.178671,-0.108101,4.236125,-0.049344,...,0.103094,0.175553,0.453542,0.181423,0.451090,0.169659,0.318208,0.184348,0.310187,0.172609
5440,1.0,748.76437,Unknown,0.725796,-1.431324,-1.052855,-0.652672,-0.518591,-0.406954,-0.311029,...,0.197692,0.275529,0.580224,0.281838,0.574945,0.269194,0.431907,0.284983,0.423283,0.272365
5441,1.0,794.79590,Unknown,1.651239,-1.204281,-0.800512,-0.373577,-0.230533,-0.111434,-0.009096,...,0.318278,0.382511,0.619537,0.388773,0.614171,0.376221,2.624637,0.391893,0.511456,0.379369


In [17]:
# split
df_join_raw = df_join_raw.rename_axis(None)
# df_join_raw = df_join_raw.iloc[:, 2:]
df_join_raw

,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,...,PD_2.43,PD_2.44,PD_2.45,PD_2.46,PD_2.47,PD_2.48,PD_2.49,PD_2.50,PD_2.51,PD_2.52
0,1.0,69.99951,Unknown,0.575329,-2.215760,-1.877019,-1.518844,-1.398838,-1.298920,-1.213064,...,-1.096866,0.333022,0.273693,0.478082,-1.012425,-1.078037,0.566607,-1.074093,-1.041128,-1.076718
1,1.0,70.04025,Unknown,2.519698,0.524517,0.862641,1.220163,1.339951,1.439686,1.525385,...,1.992662,2.054098,2.342706,2.060094,2.337721,2.048076,2.197014,2.063083,2.188817,2.051090
2,1.0,70.04151,Unknown,0.580982,-0.489549,-0.214833,0.075646,0.172970,0.254003,0.323632,...,0.520551,0.533229,0.582630,0.534187,0.581785,0.532269,0.560055,0.535142,0.559160,1.820908
3,1.0,70.04908,Unknown,2.570505,-0.107955,0.215094,0.556676,0.671123,0.766412,0.848291,...,1.365489,0.697392,2.546915,0.700091,2.183391,0.694677,2.494512,0.701435,0.730124,0.696036
4,1.0,70.06267,Unknown,1.157874,-0.151374,0.127539,0.297939,0.422454,0.521265,0.603536,...,1.125370,1.149314,2.839522,2.768711,1.226690,1.147188,1.188020,2.990083,1.187039,1.148252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,1.0,732.79951,Unknown,2.214770,-0.588492,-0.389296,-0.178671,-0.108101,4.236125,-0.049344,...,0.103094,0.175553,0.453542,0.181423,0.451090,0.169659,0.318208,0.184348,0.310187,0.172609
5440,1.0,748.76437,Unknown,0.725796,-1.431324,-1.052855,-0.652672,-0.518591,-0.406954,-0.311029,...,0.197692,0.275529,0.580224,0.281838,0.574945,0.269194,0.431907,0.284983,0.423283,0.272365
5441,1.0,794.79590,Unknown,1.651239,-1.204281,-0.800512,-0.373577,-0.230533,-0.111434,-0.009096,...,0.318278,0.382511,0.619537,0.388773,0.614171,0.376221,2.624637,0.391893,0.511456,0.379369
5442,1.0,800.81295,Unknown,1.511482,-1.036480,-0.607684,-0.154288,-0.002378,0.124103,0.232784,...,0.754284,0.812592,1.028620,1.608131,1.024227,0.807407,0.923866,0.817754,0.919121,0.810002


In [ ]:
# Filter node id (get common node id)
for encoder in encoders:
	for group_id in groups_id + ["union", "intersection"]:
		# Read common node
		common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
		print(group_id, len(common_node_id)) #, common_node_id)


group_id = "AR" # "AR" # Change
common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")

df_join_raw = df_join_raw.loc[common_node_id]
df_join_raw

In [ ]:
# get metadata
df_join_raw_metadata = df_join_raw.iloc[:, :2]
df_join_raw_metadata

In [ ]:
df_join_raw_intensity = df_join_raw.iloc[:, 3:]
df_join_raw_intensity

In [ ]:
# get 100 first AR
df_join_raw_intensity = df_join_raw.iloc[:, 363:]
df_join_raw_intensity

In [ ]:
# get groups name
groups_id = []
for item in df_join_raw_intensity.columns.values:
    group_id = item.split("_")[0]
    groups_id.append(group_id)
print(groups_id)

In [ ]:
def plot_class_distribution(labels, class_names=None):
    """
    Plot the number of instances per class.

    Parameters
    ----------
    labels : array-like
        List or array of class labels.
    class_names : dict or list, optional
        Mapping from class id to class name.
    """
    labels = np.asarray(labels)

    # Count instances per class
    classes, counts = np.unique(labels, return_counts=True)

    # X-axis labels
    if class_names is None:
        x_labels = [str(c) for c in classes]
    elif isinstance(class_names, dict):
        x_labels = [class_names[c] for c in classes]
    else:
        x_labels = [class_names[c] for c in classes]

    # Plot
    fig, ax = plt.subplots(figsize=(8, 5))

    bars = ax.bar(
        x_labels,
        counts,
        color="steelblue",
        edgecolor="black"
    )

    # Display values on top of bars
    ax.bar_label(
        bars,
        fmt="%d",
        padding=3,
        fontsize=10,
        # fontweight="bold"
    )

    ax.set_xlabel("Class")
    ax.set_ylabel("Number of instances")
    ax.set_title("Class distribution")

    ax.grid(axis="y", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()

plot_class_distribution(groups_id)

### Classification

#### Data preparation

In [ ]:
df_join_raw_intensity

In [ ]:
df_join_raw_intensity_t = df_join_raw_intensity.T
df_join_raw_intensity_t

In [ ]:
print(len(groups_id), groups_id)

In [ ]:
X = df_join_raw_intensity_t.values
X

In [ ]:
# Convertir a serie de categorías y obtener los códigos numéricos
y = pd.Series(groups_id).astype("category").cat.codes.values
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# print(y_test)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # fit only on train
X_test = scaler.transform(X_test)        # only transform on test

#### Model selection and train

In [ ]:
param_dist = {
    "n_estimators": randint(100, 500),
    "max_depth": randint(3, 15),
    "min_samples_split": randint(2, 10),
    "min_samples_leaf": randint(1, 5)
}

# Create a random forest classifier
model = RandomForestClassifier(random_state=42, n_jobs=-1)

# Use random search to find the best hyperparameters
rand_search = RandomizedSearchCV(
    model, param_distributions=param_dist,
    n_iter=10, cv=5, scoring="accuracy",
    n_jobs=-1, random_state=42)

# Select and train model
search = rand_search.fit(X_train, y_train)

# Create a variable for the best model
best_model = rand_search.best_estimator_

# Print the best hyperparameters
print("Best hyperparameters:",  rand_search.best_params_)

#### Evaluation

In [ ]:
# Evaluation
y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
recall_macro = recall_score(y_test, y_pred, average="macro")
f1_macro = f1_score(y_test, y_pred, average="macro")
classification_rep = classification_report(y_test, y_pred, output_dict=False)

print("Accuracy: ", accuracy)
print("Recall: ", recall_macro)
print("F1-score: ", f1_macro)

print(classification_rep)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
class_names = best_model.classes_
tick_marks = np.arange(len(class_names))

plt.figure()
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
# plt.colorbar()
plt.xticks(tick_marks, class_names, rotation=45, ha="right", rotation_mode="anchor")
plt.yticks(tick_marks, class_names)

# Values inside each cell
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j, i,
            format(cm[i, j], "d"),
            ha="center",
            va="center"
        )

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

### Node classification

In [ ]:
def visualize_(h, color):
    z = TSNE(n_components=2).fit_transform(h.detach().cpu().numpy())

    plt.figure(figsize=(10,10))
    plt.xticks([])
    plt.yticks([])

    plt.scatter(z[:, 0], z[:, 1], s=70, c=color, cmap="Set2")
    plt.show()

def visualize(h, color, class_names):
    z = TSNE(n_components=2, random_state=42).fit_transform(
        h.detach().cpu().numpy()
    )

    color = np.asarray(color)
    class_names = np.asarray(class_names)

    plt.figure(figsize=(10, 10))

    scatter = plt.scatter(
        z[:, 0],
        z[:, 1],
        c=color,
        cmap="tab20",
        s=70,
        alpha=0.8,
    )

    plt.xticks([])
    plt.yticks([])

    # Obtener una etiqueta por clase
    unique_classes = np.unique(color)
    handles = scatter.legend_elements()[0]

    labels = [
        class_names[color == c][0]
        for c in unique_classes
    ]

    plt.legend(
        handles,
        labels,
        title="Node types",
        loc="best"
    )

    plt.tight_layout()
    plt.show()
    
# Calculate accuracy
def accuracy(pred_y, y):
    return (pred_y == y).sum() / len(y)

In [ ]:
import torch

torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

#### Preprocessing

##### Option 1 (partial correlations)

In [ ]:
# Partial correlation

""" df_matrix = pg.pcorr(df_join_raw_intensity)
df_matrix """


from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
scaler = StandardScaler()

df_subgroup_scaled = scaler.fit_transform(df_join_raw_intensity)
lw = LedoitWolf()
lw.fit(df_subgroup_scaled)

# Matriz de covarianza regularizada
cov = lw.covariance_
std = np.sqrt(np.diag(cov))
corr = cov / np.outer(std, std)
df_matrix = pd.DataFrame(corr)
df_matrix

In [ ]:
# Edge list

threshold = np.percentile(np.abs(df_matrix), 95) # Important

df_weighted_edges = (df_matrix.where(np.triu(np.ones(df_matrix.shape), k=1).astype(bool)).stack())
df_weighted_edges = df_weighted_edges.dropna().to_frame()
df_weighted_edges.reset_index(inplace=True)
df_weighted_edges.columns = ["source", "target", "weight"]
df_weighted_edges = df_weighted_edges[df_weighted_edges["weight"].abs() >= threshold]
df_weighted_edges


In [ ]:
num_nodes = pd.concat([df_weighted_edges["source"], df_weighted_edges["target"]]).nunique()
num_nodes

##### Option 2 (K-NN)

In [ ]:
df_join_raw_intensity

In [ ]:
df_join_raw_intensity_t = df_join_raw_intensity.T
df_join_raw_intensity_t

In [ ]:
X = df_join_raw_intensity_t.values
X

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(df_join_raw_intensity_t.values)
X_scaled

In [ ]:
# similarity = cosine_similarity(X_scaled)
# similarity

In [ ]:
# Similarity matrix

k = 10 # Change

A = kneighbors_graph(
    X_scaled, # X, X_scaled
    n_neighbors=k,
    metric="cosine",
    mode="distance",
    include_self=False
)
A.toarray()

In [ ]:
# Edge list

edges = []

n = A.shape[0]

for i in range(n):
    for j in range(i + 1, n): # Solo parte superior (grafo no dirigido)
        if A[i, j] != 0:
            edges.append((i, j, A[i, j]))

print(edges)

df_weighted_edges = pd.DataFrame(edges, columns=["source", "target", "weight"])
df_weighted_edges

#### Data preparation

In [ ]:
df_weighted_edges

In [ ]:
edge_index = torch.tensor(df_weighted_edges.iloc[:, [0, 1]].values, dtype=torch.long)

data = Data(edge_index=edge_index.t().contiguous())

num_nodes = data.num_nodes
x = torch.eye(num_nodes, dtype=torch.float)
# x = torch.tensor(X_scaled, dtype=torch.float)
data.x = x

y = torch.tensor(pd.Series(groups_id).astype("category").cat.codes.values, dtype=torch.long)
data.y = y

# 2. Obtener los índices de todos los nodos en formato numpy
node_indices = torch.arange(num_nodes).numpy()
labels = data.y.numpy()

# 3. Dividir los índices usando sklearn (80% entrenamiento, 20% prueba)
# Puedes usar "stratify=labels" si quieres mantener la proporción de clases
train_idx, test_idx = train_test_split(
    node_indices, 
    test_size=0.2, 
    random_state=42, 
    stratify=labels
)

# 4. Crear las máscaras booleanas vacías (en falso)
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

# 5. Cambiar a True las posiciones correspondientes a cada conjunto
train_mask[train_idx] = True
test_mask[test_idx] = True

# 6. Asignar las máscaras al objeto Data de PyTorch Geometric
data.train_mask = train_mask
data.test_mask = test_mask
data

In [ ]:
print(data)
print()

# Gather some statistics about the graph.
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of edges: {data.num_edges}")
print(f"Average node degree: {data.num_edges / data.num_nodes:.2f}")
print(f"Number of training nodes: {data.train_mask.sum()}")
print(f"Training node label rate: {int(data.train_mask.sum()) / data.num_nodes:.2f}")
print(f"Contains isolated nodes: {data.has_isolated_nodes()}")
print(f"Contains self-loops: {data.has_self_loops()}")
print(f"Is undirected: {data.is_undirected()}")

In [ ]:
transform = T.Compose([T.ToUndirected()])#, T.AddSelfLoops()])
data = transform(data) 

print(data)
print()

# Gather some statistics about the graph.
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of edges: {data.num_edges}")
print(f"Average node degree: {data.num_edges / data.num_nodes:.2f}")
print(f"Number of training nodes: {data.train_mask.sum()}")
print(f"Training node label rate: {int(data.train_mask.sum()) / data.num_nodes:.2f}")
print(f"Contains isolated nodes: {data.has_isolated_nodes()}")
print(f"Contains self-loops: {data.has_self_loops()}")
print(f"Is undirected: {data.is_undirected()}")

In [ ]:
data.x.shape, data.x.size(-1), data.num_features

In [ ]:
num_features = data.num_features # data.x.size(-1)

# 2. Obtener el número de clases (valores únicos en el tensor y)
if data.y.dim() <= 1:
    # Para clasificación de nodos o grafos con etiquetas discretas
    num_classes = int(data.y.max()) + 1
else:
    # Para etiquetas representadas en formato one-hot o multi-label
    num_classes = data.y.size(-1)

print(f"Number of features: {num_features}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Número de nodos por clase en el conjunto de entrenamiento
unique, counts = torch.unique(
    data.y[data.train_mask],
    return_counts=True
)

classes = unique.cpu().numpy()
counts = counts.cpu().numpy()

plt.figure(figsize=(8, 5))

bars = plt.bar(
    classes.astype(str),
    counts,
    color="steelblue",
    edgecolor="black"
)

# Mostrar el número de nodos sobre cada barra
plt.bar_label(
    bars,
    fmt="%d",
    padding=3,
    fontsize=10,
    # fontweight="bold"
)

plt.xlabel("Class")
plt.ylabel("Number of training nodes")
plt.title("Training node distribution by class")

plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Número de nodos por clase en el conjunto de entrenamiento
unique, counts = torch.unique(
    data.y[data.test_mask],
    return_counts=True
)

classes = unique.cpu().numpy()
counts = counts.cpu().numpy()

plt.figure(figsize=(8, 5))

bars = plt.bar(
    classes.astype(str),
    counts,
    color="steelblue",
    edgecolor="black"
)

# Mostrar el número de nodos sobre cada barra
plt.bar_label(
    bars,
    fmt="%d",
    padding=3,
    fontsize=10,
    # fontweight="bold"
)

plt.xlabel("Class")
plt.ylabel("Number of training nodes")
plt.title("Testing node distribution by class")

plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

#### GCN

In [ ]:
# Define GCN

class GCN(torch.nn.Module):
    def __init__(self, num_features, dim_h, num_classes):
        super(GCN, self).__init__()
        torch.manual_seed(42)
        self.conv1 = GCNConv(num_features, 1024)
        self.conv2 = GCNConv(1024, 128)
        self.conv3 = GCNConv(128, dim_h)
        self.classifier = Linear(dim_h, num_classes)

    def forward(self, x, edge_index):
        # Node embeddings
        h = self.conv1(x, edge_index)
        h = h.relu()
        h = self.conv2(h, edge_index)
        h = h.relu()
        h = F.dropout(h, p=0.5, training=self.training)
        h = self.conv3(h, edge_index)
        z = h.relu()  # final GNN embedding space

        # Prediction head: linear classifier
        out = self.classifier(z)

        return out, z

# Hyperparameters
dimension = 128 # dimension of node embeddings
number_features = num_features # dimension of node features
number_classes = num_classes # number of classes

model = GCN(num_features=number_features, dim_h=dimension, num_classes=number_classes)
print(model)

In [ ]:
# Vizualization
model.eval()
out, z = model(data.x, data.edge_index)

visualize(z, data.y, groups_id)

In [ ]:
# Train

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)#, weight_decay=5e-4)

# Data for animations
embeddings = []
losses = []
accuracies = []
outputs = []
epochs = 1000

# Training loop
pbar = tqdm(range(1, epochs + 1))
for epoch in pbar:
    # Mode train
    model.train()
    
    # Clear gradients
    optimizer.zero_grad()

    # Forward pass
    out, z = model(data.x, data.edge_index)

    # Calculate loss function
    # loss = criterion(out, data.y)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])

    # Calculate accuracy
    acc = accuracy(out[data.train_mask].argmax(dim=1), data.y[data.train_mask])

    # Compute gradients
    loss.backward()

    # Tune parameters
    optimizer.step()

    # Store data for animations
    embeddings.append(z)
    losses.append(loss.item())
    accuracies.append(acc)
    outputs.append(out.argmax(dim=1))

    # Print metrics
    pbar.set_description(f"Epoch {epoch:>3} | Loss: {loss:.2f} | Acc: {acc*100:.2f}%")

    # Print metrics every 10 epochs
    """ if epoch % 10 == 0:
        print(f"Epoch {epoch:>3} | Loss: {loss:.2f} | Acc: {acc*100:.2f}%") """

In [ ]:
# Model evaluation
model.eval()
out, z = model(data.x, data.edge_index)
acc = accuracy(out[data.test_mask].argmax(dim=1), data.y[data.test_mask])
print(f"Acc: {acc*100:.2f}%")

In [ ]:
# Get node embeddings
print(z.shape)
z

In [ ]:
# Plot loss curve
plt.figure()
plt.plot(range(1, epochs + 1), losses, marker="")

# Axis labels
plt.xlabel("Epoch")
plt.ylabel("Loss")

# Title
plt.title("Training loss curve")

# Grid for better visualization
plt.grid(True)

# Display plot
plt.show()

In [ ]:
# Plot accuracy curve
plt.figure()
plt.plot(range(1, epochs + 1), accuracies, marker="")

# Axis labels
plt.xlabel("Epoch")
plt.ylabel("Loss")

# Title
plt.title("Training accuracy curve")

# Grid for better visualization
plt.grid(True)

# Display plot
plt.show()

In [ ]:
visualize(z, data.y, groups_id)

#### GAT

In [ ]:
# Define GAT
class GAT(torch.nn.Module):
    def __init__(self, num_features, dim_h, num_classes, heads=1):
        super(GAT, self).__init__()
        torch.manual_seed(42)
        # GAT layers
        """ self.conv1 = GATv2Conv(num_features, 512, heads=heads)
        self.conv2 = GATv2Conv(512 * heads, 64, heads=heads)
        self.conv3 = GATv2Conv(64 * heads, dim_h, heads=1) """
        
        """ self.conv1 = GATv2Conv(num_features, 64, heads=4)
        self.conv2 = GATv2Conv(64 * 4, 64, heads=4)
        self.conv3 = GATv2Conv(64 * 4, dim_h, heads=1) """
        
        self.conv1 = GATv2Conv(
            num_features,
            64,
            heads=4,
            dropout=0.6
        )

        self.conv2 = GATv2Conv(
            64 * 4,
            dim_h,
            heads=1,
            concat=False,
            dropout=0.6
        )
        
        # Prediction head
        self.classifier = Linear(dim_h, num_classes)

    def forward(self, x, edge_index):
        # Node embeddings
        """ h = self.conv1(x, edge_index)
        h = h.relu()
        h = self.conv2(h, edge_index)
        h = h.relu()
        h = F.dropout(h, p=0.5, training=self.training)
        h = self.conv3(h, edge_index)
        z = h.relu()  # final GNN embedding space """
        
        """ h = self.conv1(x, edge_index)
        h = F.elu(h)
        h = F.dropout(h, p=0.5, training=self.training)
        h = self.conv2(h, edge_index)
        h = F.elu(h)
        h = F.dropout(h, p=0.5, training=self.training)
        z = self.conv3(h, edge_index) """
        
        """ x = F.dropout(x, 0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, 0.6, training=self.training)
        z = self.conv2(x, edge_index) """
        
        h = self.conv1(x, edge_index)
        h = F.elu(h)
        h = F.dropout(h, p=0.6, training=self.training)
        h = self.conv2(h, edge_index)
        z = F.elu(h)
        
        # Prediction head: linear classifier
        out = self.classifier(z)
        return out, z

# Hyperparameters
dimension = 128 # node embedding dimension d'
num_features = num_features # input feature dimension d
num_classes = num_classes # number of output classes |Y|
num_heads = 2 # number of attention heads K

# Model instantiation
model = GAT(
    num_features = num_features,
    dim_h        = dimension,
    num_classes  = num_classes,
    heads        = num_heads
)
print(model)

In [ ]:
# Vizualization
model.eval()
out, z = model(data.x, data.edge_index)

visualize(z, data.y, groups_id)

In [ ]:
# Train

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)#, weight_decay=5e-4)

# Data for animations
embeddings = []
losses = []
accuracies = []
outputs = []
epochs = 1000

# Training loop
pbar = tqdm(range(1, epochs + 1))
for epoch in pbar:
    # Mode train
    model.train()

    # Clear gradients
    optimizer.zero_grad()

    # Forward pass
    out, z = model(data.x, data.edge_index)

    # Calculate loss function
    loss = criterion(out[data.train_mask], data.y[data.train_mask])

    # Calculate accuracy
    acc = accuracy(out[data.train_mask].argmax(dim=1), data.y[data.train_mask])

    # Compute gradients
    loss.backward()

    # Tune parameters
    optimizer.step()

    # Store data for animations
    embeddings.append(z)
    losses.append(loss.item())
    accuracies.append(acc)
    outputs.append(z.argmax(dim=1))

    # Print metrics
    pbar.set_description(f"Epoch {epoch:>3} | Loss: {loss:.2f} | Acc: {acc*100:.2f}%")

    # Print metrics every 10 epochs
    """ if epoch % 10 == 0:
        print(f"Epoch {epoch:>3} | Loss: {loss:.2f} | Acc: {acc*100:.2f}%") """

In [ ]:
# Model evaluation
model.eval()
out, z = model(data.x, data.edge_index)
acc = accuracy(out[data.test_mask].argmax(dim=1), data.y[data.test_mask])
print(f"Acc: {acc*100:.2f}%")

In [ ]:
model.eval()
with torch.no_grad():
    # 2. Obtener las predicciones del modelo (log_softmax o logits)
    out, z = model(data.x, data.edge_index)
    
    # 3. Convertir las salidas en las clases predichas (índice con mayor valor)
    y_pred = out.argmax(dim=1)

# 4. Filtrar solo los nodos del conjunto de prueba (Test Set)
# Usamos .cpu().numpy() porque scikit-learn trabaja con vectores de NumPy
y_test = data.y[data.test_mask].cpu().numpy()
y_pred = y_pred[data.test_mask].cpu().numpy()

accuracy = accuracy_score(y_test, y_pred)
recall_macro = recall_score(y_test, y_pred, average="macro")
f1_macro = f1_score(y_test, y_pred, average="macro")
classification_rep = classification_report(y_test, y_pred, output_dict=False)

print("Accuracy: ", accuracy)
print("Recall: ", recall_macro)
print("F1-score: ", f1_macro)

print(classification_rep)

# 5. Calcular la matriz de confusión numérica
cm = confusion_matrix(y_test, y_pred)
# print("Confusion Matrix:\n", cm)

labels = [f"{i}" for i in range(out.shape[1])] 

disp = ConfusionMatrixDisplay(confusion_matrix=cm) # , display_labels=labels)
disp.plot(cmap=plt.cm.Blues, values_format='d')

plt.title("Matriz de Confusión - Clasificación de Nodos")
plt.show()

In [ ]:
# Get node embeddings
print(z.shape)
z

In [ ]:
# Plot loss curve
plt.figure()
plt.plot(range(1, epochs + 1), losses, marker="")

# Axis labels
plt.xlabel("Epoch")
plt.ylabel("Loss")

# Title
plt.title("Training loss curve")

# Grid for better visualization
plt.grid(True)

# Display plot
plt.show()

In [ ]:
# Plot accuracy curve
plt.figure()
plt.plot(range(1, epochs + 1), accuracies, marker="")

# Axis labels
plt.xlabel("Epoch")
plt.ylabel("Loss")

# Title
plt.title("Training accuracy curve")

# Grid for better visualization
plt.grid(True)

# Display plot
plt.show()

In [ ]:
visualize(z, data.y, groups_id)